# Fase 1 - Preprocesamiento del Dataset IroSvA
Se aplica la pipeline de preprocesamiento común a todos los modelos: eliminación de duplicados, normalización de caracteres, conversión de emojis y conservación de variables lingüísticas enfáticas. Se utiliza la partición oficial train/test del dataset (Ortega-Bueno et al., 2019).

## 1. Importación de librerías

In [1]:
import pandas as pd
import re
import emoji

## 2. Carga y unión del conjunto de entrenamiento
Se combinan las tres variantes dialectales (México, España, Cuba) y se eliminan los 3 duplicados identificados en la fase de exploración.

In [2]:
df_mx_train = pd.read_csv('../data/irosva.mx.training.csv')
df_es_train = pd.read_csv('../data/irosva.es.training.csv')
df_cu_train = pd.read_csv('../data/irosva.cu.training.csv')

df_mx_train['VARIANTE'] = 'mx'
df_es_train['VARIANTE'] = 'es'
df_cu_train['VARIANTE'] = 'cu'

df_train = pd.concat([df_mx_train, df_es_train, df_cu_train], ignore_index=True)
antes = len(df_train)
df_train = df_train.drop_duplicates(subset='MESSAGE').reset_index(drop=True)
print(f"Train: {antes} → {len(df_train)} registros ({antes - len(df_train)} duplicados eliminados)")

Train: 7200 → 7197 registros (3 duplicados eliminados)


## 3. Carga y combinación del conjunto de test
Los archivos de test del dataset IroSvA separan el texto de las etiquetas reales en dos archivos distintos. Se combinan por ID antes de unir las variantes.

In [3]:
def cargar_test(variante, nombre_variante):
    df_texto = pd.read_csv(f'../data/irosva.{variante} - irosva.{variante}.test.csv')
    df_truth = pd.read_csv(f'../data/irosva.{variante}.test.truth.csv')
    # El archivo de texto tiene IS_IRONIC = '?' — se reemplaza con etiquetas reales
    df_texto = df_texto.drop(columns=['IS_IRONIC'])
    df = df_texto.merge(df_truth[['ID', 'IS_IRONIC']], on='ID')
    df['VARIANTE'] = nombre_variante
    return df

df_test = pd.concat([
    cargar_test('mx', 'mx'),
    cargar_test('es', 'es'),
    cargar_test('cu', 'cu')
], ignore_index=True)

print(f"Test: {len(df_test)} registros")
print(f"Distribución IS_IRONIC: {df_test['IS_IRONIC'].value_counts().to_dict()}")

Test: 1800 registros
Distribución IS_IRONIC: {0: 1201, 1: 599}


## 4. Función de preprocesamiento
Se conservan mayúsculas enfáticas y puntuación expresiva como señales pragmáticas de ironía (Ortega-Bueno et al., 2022). Los emojis se convierten a descripciones textuales en español para preservar su carga semántica (Qin et al., 2025). Las menciones y URLs se reemplazan por tokens neutros para evitar que el modelo aprenda identidades específicas.

In [4]:
def preprocesar(texto):
    # 1. Reemplazar URLs por token neutro
    texto = re.sub(r'http\S+|www\S+', '[URL]', texto)
    # 2. Reemplazar menciones @usuario por token neutro
    texto = re.sub(r'@\w+', '[USER]', texto)
    # 3. Convertir emojis a descripción textual en español (Qin et al., 2025)
    texto = emoji.demojize(texto, language='es')
    # 4. Normalizar caracteres repetidos: máx. 2 ocurrencias (Ortega-Bueno et al., 2022)
    texto = re.sub(r'(.)\1{2,}', r'\1\1', texto)
    # 5. Conservar puntuación enfática (¿ ¡ ! ?) como señal pragmática de ironía
    texto = re.sub(r'[^\w\s\!\?\.\,\;\:\#\[\]áéíóúüñÁÉÍÓÚÜÑ¿¡]', ' ', texto)
    # 6. Eliminar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

## 5. Aplicación del preprocesamiento y verificación con ejemplos

In [5]:
df_train['MESSAGE_CLEAN'] = df_train['MESSAGE'].apply(preprocesar)
df_test['MESSAGE_CLEAN']  = df_test['MESSAGE'].apply(preprocesar)
print(f"Preprocesamiento aplicado — Train: {df_train.shape} | Test: {df_test.shape}")

# Verificación con un ejemplo por variante
for variante in ['mx', 'es', 'cu']:
    ejemplo = df_train[df_train['VARIANTE'] == variante].iloc[0]
    print(f"\n[{variante.upper()}]")
    print(f"ORIGINAL: {ejemplo['MESSAGE']}")
    print(f"LIMPIO:   {ejemplo['MESSAGE_CLEAN']}")

Preprocesamiento aplicado — Train: (7197, 6) | Test: (1800, 6)

[MX]
ORIGINAL: Rica económicamente, pero muy pobre en objetividad.
LIMPIO:   Rica económicamente, pero muy pobre en objetividad.

[ES]
ORIGINAL: @ArmandoRuido007 @ANTI_MERMA50 @JoanTarda En vez de Joan Tarda van a llamarle “No han tarda” en callarle la boca 🤣🤣
LIMPIO:   [USER] [USER] [USER] En vez de Joan Tarda van a llamarle No han tarda en callarle la boca :cara_revolviéndose_de_la_risa::cara_revolviéndose_de_la_risa:

[CU]
ORIGINAL: magnifico
LIMPIO:   magnifico


## 6. Exportación de datos preprocesados

In [6]:
df_train.to_csv('../data/train_clean.csv', index=False)
df_test.to_csv('../data/test_clean.csv', index=False)
print("Guardado: train_clean.csv | test_clean.csv")

Guardado: train_clean.csv | test_clean.csv
